In [1]:
import import_ipynb
from Recodificacion import *


In [2]:
import warnings
import torch
import numpy as np
from skimage import io, color
from skimage.transform import resize
import lpips
from pytorch_msssim import ms_ssim

# ---------------------------
# CONFIGURACIÓN GLOBAL
# ---------------------------


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Carga única del modelo — costoso, no repetir por llamada
loss_fn = lpips.LPIPS(net='alex').to(device)
loss_fn.eval()
for p in loss_fn.parameters():
    p.requires_grad_(False)   # evita acumulación de grafo innecesaria


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\lpips\weights\v0.1\alex.pth


In [3]:

# ---------------------------
# PREPROCESAMIENTO ROBUSTO
# ---------------------------

LPIPS_MIN_SIZE = 161  # tamaño mínimo recomendado para MS-SSIM

def safe_rgb(img: np.ndarray) -> np.ndarray:
    """Garantiza salida HxWx3 uint8 o float32 según entrada."""
    if img.ndim == 2:
        img = color.gray2rgb(img)
    elif img.shape[-1] == 4:
        img = color.rgba2rgb(img)      # rgba2rgb ya devuelve [0,1] float
    return img

def load_np(path: str) -> np.ndarray:
    """
    Carga imagen y la devuelve en float32 con rango [0, 255].
    Detecta imágenes ya normalizadas a [0,1] y las reescala.
    """
    img = safe_rgb(io.imread(path))

    # rgba2rgb de skimage devuelve float en [0,1]; resto viene en [0,255] uint
    if img.dtype in (np.float32, np.float64) and img.max() <= 1.0:
        img = (img * 255.0)

    return img.astype(np.float32)

def match_size_np(
    img1: np.ndarray,
    img2: np.ndarray,
    silent: bool = False
) -> tuple[np.ndarray, np.ndarray]:
    """
    Redimensiona img2 al tamaño de img1 si difieren.
    Emite advertencia por defecto — nunca falla silenciosamente.
    """
    if img1.shape != img2.shape:
        if not silent:
            warnings.warn(
                f"Tamaños distintos: {img1.shape} vs {img2.shape}. "
                "Se redimensiona la imagen comprimida — esto puede afectar la métrica.",
                UserWarning,
                stacklevel=2,
            )
        img2 = (resize(img2, img1.shape, anti_aliasing=True) * 255.0).astype(np.float32)
    return img1, img2

def _validate_size(arr: np.ndarray, name: str) -> None:
    h, w = arr.shape[:2]
    if h < LPIPS_MIN_SIZE or w < LPIPS_MIN_SIZE:
        raise ValueError(
            f"{name} demasiado pequeña ({h}×{w}). "
            f"LPIPS/VGG requiere mínimo {LPIPS_MIN_SIZE}×{LPIPS_MIN_SIZE} px."
        )


def calcular_ms_ssim_lpips(
    img_orig_path: str,
    img_comp_path: str,
    alpha: float = 0.5,
    beta: float = 0.5,
    silent_resize: bool = False,
) -> dict:

    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha y beta deben estar en [0, 1].")

    arr_o = load_np(img_orig_path)
    arr_c = load_np(img_comp_path)
    arr_o, arr_c = match_size_np(arr_o, arr_c, silent=silent_resize)

    _validate_size(arr_o, "Imagen original")
    _validate_size(arr_c, "Imagen comprimida")

    with torch.no_grad():

        # ---- MS-SSIM — entrada en [0, 1]
        t_o = torch.from_numpy(arr_o / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)
        t_c = torch.from_numpy(arr_c / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)

        ms = ms_ssim(t_o, t_c, data_range=1.0, size_average=True).item()

        # ---- LPIPS — entrada en [-1, 1]
        lp_o = (t_o * 2.0) - 1.0
        lp_c = (t_c * 2.0) - 1.0

        lp_raw = loss_fn(lp_o, lp_c).item()

    # ---- Score combinado con valores reales ----
    # MS-SSIM: mayor = mejor  → se usa directo
    # LPIPS:   menor = mejor  → se invierte con 1/(1+lp)  → acotado en (0,1]
    lp_norm = 1.0 / (1.0 + lp_raw)  # siempre en (0,1], sin cap arbitrario

    score = alpha * ms + beta * lp_norm

    return {
        'ms_ssim':    round(ms, 6),
        'lpips_raw':  round(lp_raw, 6),   # valor real sin modificar
        'lpips_norm': round(lp_norm, 6),  # invertido matemáticamente, sin cap
        'combinado':  round(score, 6),    # siempre en (0,1]
    }

In [11]:
convert_with_ffmpeg("prueba_img/image.webp", "prueba_img/image.jpg", 70)

[FFmpeg] Convertido → prueba_img/image.jpg


True

In [12]:
print(calcular_ms_ssim_lpips("prueba_img/image.webp", "prueba_img/image.jpg", alpha=0.5, beta=0.5))

{'ms_ssim': 0.725334, 'lpips_raw': 0.351318, 'lpips_norm': 0.740018, 'combinado': 0.732676}
